# 03 Singapore GARCH Volatility Targeting

Forecast ES3.SI (SPDR STI ETF) volatility and allocate `w = min(target volatility / forecast volatility, weight cap)` to ES3. Invest the remaining `1 - w` in A35.SI (ABF Singapore Bond Index Fund), primarily Singapore government bonds.

The approximation `portfolio volatility = w * ES3 volatility` omits bond volatility and equity-bond correlation. These are reported in `sti_bond_stats` for assessment. The fixed ES3/A35 comparison uses the GARCH strategy's average in-sample ES3 weight to distinguish diversification from market timing.

| Stage | Analysis |
|---|---|
| 1 | ARCH diagnostics: Ljung-Box tests on returns and squared returns; Engle ARCH-LM |
| 2 | Fit GARCH(1,1)-Normal, GARCH(1,1)-t, and GJR-GARCH(1,1)-t in sample; select by BIC |
| 3 | Ljung-Box diagnostics on standardized residuals and their squares |
| 4 | OOS volatility evaluation against EWMA(0.94) and rolling 20-day forecasts; MSE, QLIKE, and Diebold-Mariano tests |
| 5 | Mincer-Zarnowitz regression with Newey-West standard errors; tests of intercept zero and slope one |
| 6 | Backtests against cash, EWMA/Rolling+A35, fixed ES3/A35, and buy-and-hold controls; block bootstrap Sharpe comparisons |

At the close of T, use data through T to forecast T+1 volatility and set target weights. Execute at the next open. Overnight returns accrue to old holdings; intraday returns accrue to new holdings. Default one-way costs are 10 bps for both ES3 and A35; cash earns zero.

`REFIT='monthly'` refits at each out-of-sample month start using only earlier observations in an expanding window. `REFIT=None` freezes in-sample parameters.

Input is `data/raw/sti_daily.csv` (written by `01_data_update`; keep `DOWNLOAD=False` here). `BASE_CCY='USD'` converts prices with the same-day closing USD/SGD rate (also used for opens, as an approximation). The A35 spike filter (`CLEAN_BOND`) is off because it looks at the next day's price.

Output used downstream: `results/signals/sti_weights_history.csv` (daily ES3/A35 targets within the Singapore sleeve), read by 07 and 08.

IS: 2012-08-01 to 2019-12-31. OOS: 2020-01-01 to 2026-09-30. Live tracking begins on 2026-10-01. Outputs are saved under `results/tables/sti_*.csv`, `results/figures/sti_*.png`, and `results/signals/`.


**Which numbers to quote:** the tables in this notebook are research diagnostics. Quote final-portfolio performance from `results/tables/final_metrics.csv` (made by 07, also shown in `results/Report.html`).


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from arch import arch_model
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

# Locate the project root from the root itself or any subfolder, then share 00_engine via portfolio_accounting.
ROOT = Path.cwd().resolve()
while not (ROOT / 'pipeline' / '00_engine.ipynb').is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from portfolio_accounting import ENGINE as _engine

# Same sample split and performance metrics as the US strategy
IS_START, IS_END = _engine['IS_START'], _engine['IS_END']
OOS_START, OOS_END = _engine['OOS_START'], _engine['OOS_END']
metrics, fmt = _engine['metrics'], _engine['fmt']

TICKER = 'ES3.SI'      # SPDR STI ETF
BOND_TICKER = 'A35.SI' # ABF Singapore Bond Index Fund
FX_TICKER = 'SGD=X'    # USD/SGD quoted as SGD per 1 USD
SLEEVE_WEIGHT = 0.10   # initial Singapore share of the total portfolio (illustration in sti_latest_weight only)
STI_CSV = ROOT / 'data' / 'raw' / 'sti_daily.csv'
TABLE_DIR = ROOT / 'results' / 'tables'
FIG_DIR = ROOT / 'results' / 'figures'
SIGNAL_DIR = ROOT / 'results' / 'signals'

MODEL_SPECS = {
    'GARCH(1,1)-Normal': dict(vol='GARCH', p=1, o=0, q=1, dist='normal'),
    'GARCH(1,1)-t':      dict(vol='GARCH', p=1, o=0, q=1, dist='t'),
    'GJR-GARCH(1,1)-t':  dict(vol='GARCH', p=1, o=1, q=1, dist='t'),
}
EWMA_LAMBDA = 0.94     # RiskMetrics
ROLL_WINDOW = 20

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 0. Download and load data


In [ ]:
def _flatten(df):
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = df.columns.get_level_values(0)
    return df


def _ohlc_frame(px, prefix=''):
    return pd.DataFrame({
        f'{prefix}open': px['Open'], f'{prefix}high': px['High'], f'{prefix}low': px['Low'],
        f'{prefix}close': px['Close'], f'{prefix}adj_close': px['Adj Close'], f'{prefix}volume': px['Volume'],
    })


def download_sti(output_path=STI_CSV, start='2011-10-01'):
    """Download ES3.SI and A35.SI daily prices plus USD/SGD; save a flat CSV with one row per ES3 trading day."""
    import yfinance as yf
    get = lambda t: _flatten(yf.download(t, start=start, auto_adjust=False, progress=False))
    px, bond, fx = get(TICKER), get(BOND_TICKER), get(FX_TICKER)
    for name, d in ((TICKER, px), (BOND_TICKER, bond), (FX_TICKER, fx)):
        if d.empty:
            raise ValueError(f'{name}: download returned no data; nothing saved.')
    out = _ohlc_frame(px)
    # Align A35 to the ES3 calendar; A35 gaps stay blank and are treated as unchanged prices when loading
    out = out.join(_ohlc_frame(bond, 'bond_').reindex(out.index))
    fx_close = fx['Close']
    # FX and SGX calendars differ: use the latest FX quote on or before each date
    out['usdsgd'] = fx_close.reindex(out.index.union(fx_close.index)).ffill().reindex(out.index)
    out.index.name = 'Date'
    path = Path(output_path).expanduser().resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(path)
    print('Saved:', path, '|', out.index[0].date(), '~', out.index[-1].date(), '|', len(out), 'rows')
    return out


def load_sti(csv_path=None, base_ccy='USD', clean_bond=False):
    """Load dividend-adjusted open/close prices in base_ccy.

    Returns (O, C) for ES3, (OB, CB) for A35 (None if an old CSV has no A35 columns), and a data-quality dict."""
    path = Path(csv_path or STI_CSV).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f'{path} not found. Run 1_data/01_data_update first to download the data.')
    df = pd.read_csv(path, index_col=0, parse_dates=True).sort_index()
    df = df[~df.index.duplicated(keep='last')]
    n_raw = len(df)
    # SGX closes at 17:00; a row for today downloaded before 17:30 Singapore time holds intraday prices, so drop it
    now_sgt = pd.Timestamp.now(tz='Asia/Singapore')
    if len(df) and df.index[-1].date() >= now_sgt.date() and now_sgt.hour * 60 + now_sgt.minute < 17 * 60 + 30:
        print(f'⚠️ Dropped unfinished session {df.index[-1].date()} (Singapore time {now_sgt:%H:%M}, market not yet closed)')
        df = df.iloc[:-1]
    df = df.dropna(subset=['open', 'close', 'adj_close'])
    df = df[(df['open'] > 0) & (df['close'] > 0)]
    if base_ccy.upper() == 'USD':
        fx = df['usdsgd'].ffill()
    elif base_ccy.upper() == 'SGD':
        fx = pd.Series(1.0, index=df.index)
    else:
        raise ValueError("base_ccy must be 'USD' or 'SGD'")

    factor = df['adj_close'] / df['close']          # dividend adjustment factor, also applied to opens
    O = df['open'] * factor / fx
    C = df['adj_close'].astype(float) / fx
    quality = {
        'File': str(path), 'Period': f'{C.index[0].date()} ~ {C.index[-1].date()}',
        'Raw rows': n_raw, 'Valid rows': len(C),
        'ES3 share of zero-return days': round(float((C.pct_change() == 0).mean()), 4),
        'Reporting currency': base_ccy.upper(),
    }

    OB = CB = None
    if {'bond_open', 'bond_close', 'bond_adj_close'}.issubset(df.columns):
        cb_sgd = df['bond_adj_close'].astype(float).ffill()
        ob_sgd = df['bond_open'] * df['bond_adj_close'] / df['bond_close']
        bad = ob_sgd.isna() | (df['bond_open'] <= 0) | df['bond_adj_close'].isna()
        # Optional cleaning (off by default because it looks at the next day): early A35 trading was thin and has
        # spikes that reverse the next day. Rule: |r_t| > 2.5%, next day reverses, and the two-day total is under half
        # of |r_t| -> treat as a bad quote and carry the previous price.
        n_spikes = 0
        if clean_bond:
            for _ in range(3):
                r_b = cb_sgd.pct_change()
                nxt = r_b.shift(-1)
                spike = (r_b.abs() > 0.025) & (r_b * nxt < 0) & ((r_b + nxt).abs() < 0.5 * r_b.abs())
                if not spike.any():
                    break
                n_spikes += int(spike.sum())
                cb_sgd = cb_sgd.mask(spike).ffill()
                bad = bad | spike
        ob_sgd = ob_sgd.where(~bad, cb_sgd)          # missing/invalid A35 quote: treat as unchanged price
        if cb_sgd.isna().any():
            raise ValueError(f'{BOND_TICKER} has no data before {cb_sgd[cb_sgd.isna()].index[0].date()}.')
        OB, CB = (ob_sgd / fx).rename(BOND_TICKER), (cb_sgd / fx).rename(BOND_TICKER)
        quality['A35 bad quotes cleaned'] = n_spikes
        quality['A35 missing or invalid quote days (ES3 calendar)'] = int(bad.sum())
        quality['A35 share of zero-return days'] = round(float((CB.pct_change() == 0).mean()), 4)
    else:
        print(f'⚠️ CSV has no {BOND_TICKER} data; run 1_data/01_data_update again. Cash replaces the bond for this run.')
    return O.rename(TICKER), C.rename(TICKER), OB, CB, quality


def log_returns_pct(C):
    """Log returns in percent (x100); GARCH estimation is numerically more stable on this scale."""
    return (100 * np.log(C).diff()).dropna().rename('r')

## 1. ARCH effect tests

Significant autocorrelation in squared returns indicates volatility clustering and motivates conditional volatility modeling.


In [ ]:
def arch_effect_tests(r, lags=(5, 10, 20), arch_lags=10):
    rows = []
    lb_r = acorr_ljungbox(r, lags=list(lags), return_df=True)
    lb_r2 = acorr_ljungbox(r ** 2, lags=list(lags), return_df=True)
    for lag in lags:
        rows.append({'Test': f'Ljung-Box on r, lag={lag}', 'Statistic': lb_r.loc[lag, 'lb_stat'], 'p-value': lb_r.loc[lag, 'lb_pvalue']})
    for lag in lags:
        rows.append({'Test': f'Ljung-Box on r², lag={lag}', 'Statistic': lb_r2.loc[lag, 'lb_stat'], 'p-value': lb_r2.loc[lag, 'lb_pvalue']})
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', FutureWarning)
        lm, lm_p, _, _ = het_arch(r - r.mean(), nlags=arch_lags)
    rows.append({'Test': f'Engle ARCH-LM, lag={arch_lags}', 'Statistic': lm, 'p-value': lm_p})
    table = pd.DataFrame(rows)
    table['Result (5%)'] = np.where(table['p-value'] < 0.05, 'Reject H0', 'Do not reject H0')
    return table

## 2. In-sample model estimation and BIC selection

Persistence is `alpha + beta + gamma/2` for GJR, approximating negative shocks as active half the time. Half-life is `ln(0.5) / ln(persistence)`, measured in trading days.


In [ ]:
def fit_model(r, spec):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return arch_model(r, mean='Constant', **spec).fit(disp='off')


def summarize_fit(name, res):
    p = res.params
    alpha, beta, gamma = p.get('alpha[1]', np.nan), p.get('beta[1]', np.nan), p.get('gamma[1]', 0.0)
    pers = alpha + beta + gamma / 2
    half_life = np.log(0.5) / np.log(pers) if 0 < pers < 1 else np.inf
    uncond_ann = np.sqrt(p['omega'] / (1 - pers) * 252) / 100 if pers < 1 else np.nan
    return {
        'Model': name, 'ω': p['omega'], 'α': alpha, 'γ': p.get('gamma[1]', np.nan), 'β': beta,
        'ν (t d.o.f.)': p.get('nu', np.nan), 'Persistence': pers, 'Half-life (days)': half_life,
        'Long-run annualized volatility': uncond_ann, 'LogLik': res.loglikelihood, 'AIC': res.aic, 'BIC': res.bic,
        'Converged': res.convergence_flag == 0,
    }


def compare_models(r_is):
    fits = {name: fit_model(r_is, spec) for name, spec in MODEL_SPECS.items()}
    table = pd.DataFrame([summarize_fit(n, f) for n, f in fits.items()]).set_index('Model')
    ok = table[table['Converged'] & (table['Persistence'] < 1)]
    if ok.empty:
        raise RuntimeError('No model converged to a stationary fit; check the data.')
    best = ok['BIC'].idxmin()
    return fits, table, best


def param_table(res):
    return pd.DataFrame({'Estimate': res.params, 'Standard error': res.std_err, 't-stat': res.tvalues, 'p-value': res.pvalues})

## 3. Residual diagnostics

Standardized residuals `z = epsilon / sigma` and their squares should have no significant residual autocorrelation if volatility clustering is adequately captured.


In [ ]:
def residual_diagnostics(res, lags=(10, 20)):
    z = res.std_resid.dropna()
    lb_z = acorr_ljungbox(z, lags=list(lags), return_df=True)
    lb_z2 = acorr_ljungbox(z ** 2, lags=list(lags), return_df=True)
    rows = []
    for lag in lags:
        rows.append({'Series': 'z', 'lag': lag, 'Statistic': lb_z.loc[lag, 'lb_stat'], 'p-value': lb_z.loc[lag, 'lb_pvalue']})
        rows.append({'Series': 'z²', 'lag': lag, 'Statistic': lb_z2.loc[lag, 'lb_stat'], 'p-value': lb_z2.loc[lag, 'lb_pvalue']})
    table = pd.DataFrame(rows)
    moments = {'z Skewness': round(float(z.skew()), 4), 'z Excess kurtosis': round(float(z.kurt()), 4)}
    return table, moments

## 4. One-step forecasts and out-of-sample evaluation

Each forecast indexed by t is formed after the close of t for the variance of t+1.
- GARCH uses in-sample parameters in sample and monthly expanding-window refits out of sample, or frozen in-sample parameters when configured.
- EWMA follows `h_t = lambda * h_(t-1) + (1-lambda) * r_t^2`.
- Rolling forecasts use the mean of the previous 20 daily squared returns.

The evaluation target is `r_(t+1)^2`, a noisy proxy for realized variance. QLIKE is `ln(h) + r^2/h` and penalizes underestimation more heavily. The Diebold-Mariano test evaluates whether the mean loss difference `L(GARCH) - L(benchmark)` is zero with Newey-West standard errors. A negative t statistic favors GARCH.


In [ ]:
def garch_forecast_fixed(r, spec, params, start=None):
    """Filter with fixed parameters; for each origin t return the variance forecast for t+1."""
    fixed = arch_model(r, mean='Constant', **spec).fix(params)
    f = fixed.forecast(horizon=1, start=start or r.index[0], reindex=False)
    return f.variance['h.1']


def garch_forecast_refit(r, spec, oos_start, freq='MS', verbose=True):
    """Refit at each OOS month start on earlier data only, then use those parameters for next-day forecasts that month."""
    pieces, n_fit = [], 0
    months = pd.date_range(pd.Timestamp(oos_start), r.index[-1], freq=freq)
    for m0 in months:
        m1 = m0 + pd.tseries.frequencies.to_offset(freq)
        train = r.loc[:m0 - pd.Timedelta(days=1)]
        res = fit_model(train, spec)
        n_fit += 1
        h = garch_forecast_fixed(r, spec, res.params, start=m0)
        pieces.append(h.loc[(h.index >= m0) & (h.index < m1)])
    if verbose:
        print(f'OOS refits: {n_fit} (monthly, expanding window)')
    return pd.concat(pieces)


def benchmark_forecasts(r):
    r2 = r ** 2
    return {
        'EWMA(0.94)': r2.ewm(alpha=1 - EWMA_LAMBDA, adjust=False).mean(),
        f'Rolling {ROLL_WINDOW}d': r2.rolling(ROLL_WINDOW).mean(),
    }


def _hac_mean_test(d, maxlags=10):
    res = sm.OLS(np.asarray(d, float), np.ones(len(d))).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    return float(res.params[0]), float(res.tvalues[0]), float(res.pvalues[0])


def evaluate_forecasts(r, forecasts, start, end=None, ref='GARCH'):
    """forecasts: {name: forecast series indexed by origin}; evaluate on target dates within [start, end]."""
    y = (r ** 2).rename('y')
    F = pd.DataFrame({k: v.reindex(r.index).shift(1) for k, v in forecasts.items()})   # align to the target date t+1
    data = pd.concat([y, F], axis=1).loc[start:end].dropna()
    loss_mse = {k: (data['y'] - data[k]) ** 2 for k in forecasts}
    loss_qlike = {k: np.log(data[k]) + data['y'] / data[k] for k in forecasts}
    rows = []
    for k in forecasts:
        row = {'Model': k, 'MSE': loss_mse[k].mean(), 'QLIKE': loss_qlike[k].mean()}
        if k != ref:
            for lname, L in (('MSE', loss_mse), ('QLIKE', loss_qlike)):
                m, t, p = _hac_mean_test(L[ref] - L[k])
                row[f'DM {lname}: t'] = t
                row[f'DM {lname}: p'] = p
        rows.append(row)
    return pd.DataFrame(rows).set_index('Model'), data

## 5. Mincer-Zarnowitz regression

Regress next-day squared returns on an intercept and forecast variance. Unbiased forecasts imply an intercept of zero and slope of one, tested jointly with a Wald test. Squared returns are noisy, so a low R-squared is expected. Assess whether the slope is near one and compare explanatory power across models.


In [ ]:
def mincer_zarnowitz(data, names, maxlags=10):
    rows = []
    for k in names:
        X = sm.add_constant(data[[k]].rename(columns={k: 'forecast'}))
        res = sm.OLS(data['y'], X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
        wald = res.wald_test('const = 0, forecast = 1', use_f=True, scalar=True)
        rows.append({
            'Model': k, 'a': res.params['const'], 't(a)': res.tvalues['const'],
            'b': res.params['forecast'], 't(b=0)': res.tvalues['forecast'],
            't(b=1)': (res.params['forecast'] - 1) / res.bse['forecast'],
            'Joint Wald F (a=0,b=1)': float(wald.statistic), 'Wald p-value': float(wald.pvalue),
            'R²': res.rsquared,
        })
    return pd.DataFrame(rows).set_index('Model')

## 6. ES3 and A35 volatility targeting backtest

`w_t = min(sigma_target / forecast_sigma_(t+1), W_MAX)` is the ES3 target; A35 receives the remainder. The default target is ES3's realized annualized in-sample volatility. Target changes within `REBAL_BAND` do not trigger a target update.

Controls include GARCH plus cash, EWMA/Rolling20day plus A35, fixed ES3/A35, and buy-and-hold ES3 or A35. The fixed allocation uses the GARCH strategy's average **in-sample** weight and therefore removes its time-varying allocation decision.


In [ ]:
def vol_target_weights(h, target_ann, w_max=1.0, band=0.10):
    """h: daily variance forecast (percent²). Returns the ES3 target set at each close t, executed at the t+1 open."""
    sig_ann = np.sqrt(h) * np.sqrt(252) / 100
    raw = np.minimum(target_ann / sig_ann, w_max)
    out, cur = [], np.nan
    for x in raw.values:
        if np.isfinite(x) and (not np.isfinite(cur) or abs(x - cur) > band):
            cur = x
        out.append(cur)
    return pd.Series(out, index=h.index, name='weight')


def backtest_two_asset(O, C, OB, CB, w_target, start, cost_bps=10, bond_cost_bps=10, use_bond=True):
    """ES3 target weight w; the remaining 1-w holds A35 (or zero-return, zero-cost cash if use_bond=False or no bond data).
    Target set at close T is executed at the T+1 open: overnight return accrues to old holdings, intraday return to new
    holdings; costs are charged on each leg. Returns equity, executed ES3 weight, executed bond weight, ES3 turnover."""
    idx = C.index
    w_target = w_target.reindex(idx)
    bond_on = use_bond and OB is not None
    o, c, wt = O.values, C.values, w_target.values
    ob = OB.reindex(idx).values if bond_on else np.ones(len(idx))
    cb = CB.reindex(idx).values if bond_on else np.ones(len(idx))
    ce, cbps = cost_bps / 1e4, (bond_cost_bps / 1e4 if bond_on else 0.0)
    k0 = max(idx.searchsorted(pd.Timestamp(start)), 1)
    nav, h_e, h_b = 1.0, 0.0, 0.0                  # start fully in cash
    eq, held_e, held_b, turn = {idx[k0 - 1]: 1.0}, {}, {}, {}
    for k in range(k0, len(idx)):
        on_e, on_b = o[k] / c[k - 1] - 1, ob[k] / cb[k - 1] - 1
        id_e, id_b = c[k] / o[k] - 1, cb[k] / ob[k] - 1
        g = 1 + h_e * on_e + h_b * on_b             # overnight: old holdings
        nav *= g
        d_e, d_b = h_e * (1 + on_e) / g, h_b * (1 + on_b) / g
        if np.isfinite(wt[k - 1]):
            t_e = wt[k - 1]
            t_b = (1 - t_e) if bond_on else 0.0
        else:
            t_e, t_b = d_e, d_b
        trade_e, trade_b = abs(t_e - d_e), abs(t_b - d_b)
        nav *= 1 - trade_e * ce - trade_b * cbps     # costs of the opening rebalance
        g = 1 + t_e * id_e + t_b * id_b              # intraday: new holdings
        nav *= g
        h_e, h_b = t_e * (1 + id_e) / g, t_b * (1 + id_b) / g
        eq[idx[k]], held_e[idx[k]], held_b[idx[k]], turn[idx[k]] = nav, t_e, t_b, trade_e
    return pd.Series(eq), pd.Series(held_e), pd.Series(held_b), pd.Series(turn)


def bond_stats(C, CB, periods):
    """A35 risk and its correlation with ES3 (checks the approximation sleeve vol ≈ w × ES3 vol)."""
    re, rb = C.pct_change(), CB.pct_change()
    rows = []
    for label, a, b in periods:
        x, y = re.loc[a:b], rb.loc[a:b]
        ok = x.notna() & y.notna()
        x, y = x[ok], y[ok]
        worst = x <= x.quantile(0.05)
        rows.append({
            'Period': label,
            'ES3 annualized vol': x.std() * np.sqrt(252),
            'A35 annualized vol': y.std() * np.sqrt(252),
            'ES3-A35 daily return correlation': x.corr(y),
            'ES3 worst 5% days: ES3 mean return': x[worst].mean(),
            'ES3 worst 5% days: A35 mean return': y[worst].mean(),
        })
    return pd.DataFrame(rows).set_index('Period')


def block_bootstrap_sharpe_diff(ra, rb, n_boot=2000, block=20, seed=42):
    """Paired moving-block bootstrap test of whether Sharpe(a) - Sharpe(b) differs from 0."""
    x = np.column_stack([ra, rb])
    n = len(x)
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n / block))
    sharpe = lambda m: m.mean(0) / m.std(0, ddof=1) * np.sqrt(252)
    obs = sharpe(x)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        starts = rng.integers(0, n, n_blocks)
        ids = ((starts[:, None] + np.arange(block)) % n).ravel()[:n]
        s = sharpe(x[ids])
        diffs[i] = s[0] - s[1]
    d_obs = obs[0] - obs[1]
    p = float(np.mean(np.abs(diffs - diffs.mean()) >= abs(d_obs)))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {'Sharpe (strategy)': obs[0], 'Sharpe (comparator)': obs[1], 'Difference': d_obs,
            '95% CI lower': lo, '95% CI upper': hi, 'p-value': p}


def strategy_table(curves, weights, bond_weights, turns, periods):
    rows = []
    for name, eq in curves.items():
        for label, a, b in periods:
            e = eq.loc[a:b]
            if len(e) < 30:
                continue
            m = fmt(metrics(e))
            t = turns[name].loc[a:b]
            rows.append({'Strategy': name, 'Period': label, **m,
                         'Avg ES3 weight': round(float(weights[name].loc[a:b].mean()), 3),
                         'Avg bond weight': round(float(bond_weights[name].loc[a:b].mean()), 3),
                         'ES3 annual turnover': round(float(t.sum() / (len(t) / 252)), 2)})
    return pd.DataFrame(rows)

## Charts and saving


In [ ]:
def plot_all(r, h_garch, main_name, weights, bond_weights, curves, save, oos_start, bond_label):
    figs = {}
    sig = (np.sqrt(h_garch.reindex(r.index).shift(1)) * np.sqrt(252) / 100)   # Align to the forecast target date
    absr = r.abs() * np.sqrt(252) / 100
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(absr.index, absr, lw=0.4, alpha=0.4, label='|r| annualized')
    ax.plot(sig.index, sig, lw=1.2, label='GARCH forecast annualized vol')
    ax.axvline(pd.Timestamp(oos_start), color='gray', ls='--', lw=1)
    ax.set_title('ES3 volatility clustering and GARCH one-step forecast'); ax.legend(); fig.tight_layout()
    figs['sti_garch_vol'] = fig

    fig, ax = plt.subplots(figsize=(11, 3.2))
    we, wb = weights[main_name], bond_weights[main_name]
    ax.stackplot(we.index, we.values, wb.values, labels=['ES3', bond_label], alpha=0.85)
    ax.axvline(pd.Timestamp(oos_start), color='gray', ls='--', lw=1)
    ax.set_ylim(0, 1.02); ax.set_title(f'Singapore sleeve weights ({main_name})')
    ax.legend(loc='lower left'); fig.tight_layout()
    figs['sti_weight'] = fig

    fig, ax = plt.subplots(figsize=(11, 4.5))
    show = [n for n in curves if not n.startswith(('EWMA', 'Rolling'))]
    for name in show:
        eq = curves[name]
        ax.plot(eq.index, eq / eq.iloc[0], lw=1.6 if name == main_name else 1.0, label=name)
    ax.axvline(pd.Timestamp(oos_start), color='gray', ls='--', lw=1)
    ax.set_yscale('log'); ax.set_title('Equity curves (log scale)'); ax.legend(fontsize=8); fig.tight_layout()
    figs['sti_equity'] = fig

    if save:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        for name, f in figs.items():
            f.savefig(FIG_DIR / f'{name}.png', dpi=150)
    plt.show()
    return figs


def save_tables(tables):
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    for name, t in tables.items():
        t.to_csv(TABLE_DIR / f'{name}.csv', encoding='utf-8-sig')
    print('Tables saved to', TABLE_DIR)

## Main workflow


In [ ]:
def main(csv_path=None, download=False, base_ccy='USD', refit='monthly', w_max=1.0,
         band=0.10, cost_bps=10, bond_cost_bps=10, target_ann=None, n_boot=2000, save=True, verbose=True,
         clean_bond=False):
    if download:
        download_sti(csv_path or STI_CSV)
    O, C, OB, CB, quality = load_sti(csv_path, base_ccy, clean_bond)
    has_bond = OB is not None
    bond_label = 'A35' if has_bond else 'Cash'
    print('Data quality:'); display(pd.Series(quality).to_frame('Value'))
    if quality['ES3 share of zero-return days'] > 0.05:
        print('Note: ES3 has many zero-return days, mostly 2012-2017 (coarse price ticks); disclose as a data limitation.')
    r = log_returns_pct(C)
    r_is = r.loc[:IS_END]

    print('\n① ARCH effect tests (in-sample)')
    t_arch = arch_effect_tests(r_is); display(t_arch.round(4))

    print('\n② In-sample model comparison')
    fits, t_models, best = compare_models(r_is)
    display(t_models.round(4))
    best_res, best_spec = fits[best], MODEL_SPECS[best]
    print(f'Selected by BIC: {best}')
    t_params = param_table(best_res); display(t_params.round(4))

    print('\n③ Residual diagnostics (selected model)')
    t_resid, moments = residual_diagnostics(best_res)
    display(t_resid.round(4)); print(moments)

    print('\n④ Volatility forecasts')
    h_is = garch_forecast_fixed(r, best_spec, best_res.params)
    h_oos = garch_forecast_refit(r, best_spec, OOS_START, verbose=verbose) if refit else h_is.loc[OOS_START:]
    h_garch = pd.concat([h_is.loc[:pd.Timestamp(OOS_START) - pd.Timedelta(days=1)], h_oos]).rename('GARCH')
    forecasts = {'GARCH': h_garch, **benchmark_forecasts(r)}
    t_eval_oos, data_oos = evaluate_forecasts(r, forecasts, OOS_START, OOS_END)
    print('OOS forecast losses (lower is better) and DM tests (t < 0 favours GARCH)'); display(t_eval_oos.round(4))

    print('\n⑤ Mincer-Zarnowitz regression (OOS, Newey-West standard errors)')
    t_mz = mincer_zarnowitz(data_oos, list(forecasts)); display(t_mz.round(4))

    print(f'\n⑥ Volatility-targeting strategy (remainder held in: {bond_label})')
    if target_ann is None:
        target_ann = float(r.loc[IS_START:IS_END].std() * np.sqrt(252) / 100)
    print(f'Target annualized vol σ_target = {target_ann:.2%} (in-sample ES3 realized vol) | cap {w_max} | rebalance band {band} '
          f'| costs ES3 {cost_bps} bps / bond {bond_cost_bps} bps')
    w_garch = vol_target_weights(h_garch, target_ann, w_max, band)
    w_bar = float(w_garch.loc[IS_START:IS_END].mean())          # in-sample information only
    ones = pd.Series(1.0, index=C.index)
    main_name = f'GARCH vol target + {bond_label}'
    static_name = f'Fixed ES3 {w_bar:.0%} + {bond_label}'
    specs = {
        main_name: (w_garch, True),
        'GARCH vol target + Cash': (w_garch, False),
        f'EWMA vol target + {bond_label}': (vol_target_weights(forecasts['EWMA(0.94)'], target_ann, w_max, band), True),
        f'Rolling {ROLL_WINDOW}d vol target + {bond_label}': (vol_target_weights(forecasts[f'Rolling {ROLL_WINDOW}d'], target_ann, w_max, band), True),
        static_name: (ones * w_bar, True),
        'Buy and hold ES3': (ones, False),
    }
    if has_bond:
        specs['Buy and hold A35'] = (ones * 0.0, True)
    curves, weights, bond_weights, turns = {}, {}, {}, {}
    for name, (w, use_bond) in specs.items():
        curves[name], weights[name], bond_weights[name], turns[name] = backtest_two_asset(
            O, C, OB, CB, w, IS_START, cost_bps, bond_cost_bps, use_bond)
    periods = [('IS', IS_START, IS_END), ('OOS', OOS_START, OOS_END), ('OOS ex-2020', '2021-01-01', OOS_END)]
    t_strategy = strategy_table(curves, weights, bond_weights, turns, periods); display(t_strategy)

    t_bond = None
    if has_bond:
        t_bond = bond_stats(C, CB, periods)
        print('A35 risk and equity-bond correlation'); display(t_bond.round(4))

    def oos_returns(name):
        return curves[name].loc[OOS_START:OOS_END].pct_change().dropna()
    comparisons = ['Buy and hold ES3', static_name] + (['GARCH vol target + Cash'] if has_bond else [])
    boot_rows, boot_index = [], []
    for other in comparisons:
        ra, rb = oos_returns(main_name).align(oos_returns(other), join='inner')
        boot_rows.append(block_bootstrap_sharpe_diff(ra.values, rb.values, n_boot=n_boot))
        boot_index.append(f'{main_name} vs {other} (OOS)')
    t_boot = pd.DataFrame(boot_rows, index=boot_index)
    print('Block bootstrap test of Sharpe differences'); display(t_boot.round(4))

    # Latest signal: target set after the last close, for the next trading day
    last = h_garch.index[-1]
    w_last = float(w_garch.iloc[-1])
    latest = pd.DataFrame([{
        'Decision date (close)': last.date(),
        'Forecast next-day annualized vol': round(float(np.sqrt(h_garch.iloc[-1]) * np.sqrt(252) / 100), 4),
        'Target annualized vol': round(target_ann, 4),
        'Within sleeve: ES3 weight': round(w_last, 4),
        f'Within sleeve: {bond_label} weight': round(1 - w_last, 4),
        f'Of total at initial {SLEEVE_WEIGHT:.0%}: ES3': round(SLEEVE_WEIGHT * w_last, 4),
        f'Of total at initial {SLEEVE_WEIGHT:.0%}: {bond_label}': round(SLEEVE_WEIGHT * (1 - w_last), 4),
    }])
    print('\nLatest signal'); display(latest)

    figs = plot_all(r, h_garch, main_name, weights, bond_weights, curves, save, OOS_START, bond_label)
    tables = {
        'sti_arch_tests': t_arch, 'sti_model_compare': t_models, 'sti_params': t_params,
        'sti_residual_diagnostics': t_resid, 'sti_forecast_eval_oos': t_eval_oos,
        'sti_mincer_zarnowitz': t_mz, 'sti_strategy_metrics': t_strategy, 'sti_sharpe_bootstrap': t_boot,
    }
    if t_bond is not None:
        tables['sti_bond_stats'] = t_bond
    if save:
        save_tables(tables)
        SIGNAL_DIR.mkdir(parents=True, exist_ok=True)
        latest.to_csv(SIGNAL_DIR / 'sti_latest_weight.csv', index=False, encoding='utf-8-sig')
        print('Latest signal saved to', SIGNAL_DIR / 'sti_latest_weight.csv')
        # Consumed by portfolio_accounting.py: within-sleeve targets set after each Singapore close.
        hist = pd.DataFrame({'es3_weight': w_garch,
                             'a35_weight': (1 - w_garch) if has_bond else 0.0}).dropna()
        hist.index.name = 'date'
        hist.round(6).to_csv(SIGNAL_DIR / 'sti_weights_history.csv', encoding='utf-8-sig')
        print('Daily weights saved to', SIGNAL_DIR / 'sti_weights_history.csv', f'({len(hist)} rows)')
    print('\nSTI GARCH stage finished.')
    return {'quality': quality, 'model': best, 'fit': best_res, 'forecasts': forecasts,
            'weights': weights, 'bond_weights': bond_weights, 'curves': curves,
            'tables': tables, 'latest': latest, 'figures': figs}

In [ ]:
# ===== Settings: edit here, then Run All =====
DOWNLOAD = False        # prices come from 1_data/01_data_update; keep False
CSV_PATH = None         # None = data/raw/sti_daily.csv
BASE_CCY = 'USD'        # 'USD' (same reporting currency as the US strategy) or 'SGD'
REFIT = 'monthly'       # 'monthly' = monthly expanding-window OOS refits; None = freeze in-sample parameters
W_MAX = 1.0             # ES3 weight cap (1 = no leverage)
REBAL_BAND = 0.10       # update the target only if it moves by more than this
COST_BPS = 10           # ES3 one-way cost (bps)
BOND_COST_BPS = 10      # A35 one-way cost (bps)
CLEAN_BOND = False      # A35 spike cleaning; off because it uses next-day prices
SAVE = True             # save tables, figures and signals to results/

results_sti = main(csv_path=CSV_PATH, download=DOWNLOAD, base_ccy=BASE_CCY, refit=REFIT,
                   w_max=W_MAX, band=REBAL_BAND, cost_bps=COST_BPS, bond_cost_bps=BOND_COST_BPS, save=SAVE,
                   clean_bond=CLEAN_BOND)